In [ ]:
# Recalculate park benefits and payback outcomes under alternative inferred-step assumptions.
from __future__ import annotations

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

VISIT_FILE_1 = r"data/restricted/mobility/mobility_part_0.csv"
VISIT_FILE_2 = r"data/restricted/mobility/mobility_part_1.csv"
PARK_COST_FILE = r"data/restricted/park_costs/park_costs_2019_2024.xlsx"

OUTPUT_DIR = Path(
    r"outputs/sensitivity_steps_payback_clean"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPANSION_FACTOR = 134.38
JPY_PER_STEP = 0.04

SCENARIOS = [1.0, 0.8, 0.9, 1.1, 1.2]

COL_OSM_ID_VISIT = "osm_id"
COL_AREA_VISIT = "area"
COL_PARK_CLASS_VISIT = "park_class"
COL_PARK_CLASS_NAME_VISIT = "park_class_name"
COL_STEPS_VISIT = "steps"

COL_OSM_ID_COST = "osm_id"
COL_LAND_COST = "land_price_1year"
COL_UNIT_MAINT = "unit_maintenance_yen_per_m2_2024"

COL_CONSTRUCTION_COST = None

COL_PARK_TYPE_LABEL_COST = None

PAYBACK_BINS = [0, 5, 20, 50, np.inf]
PAYBACK_LABELS = ["0-5y", "5-20y", "20-50y", ">50y"]

def standardize_osm_id(s: pd.Series) -> pd.Series:
    """
    Standardize osm_id to comparable string format.
    Removes trailing '.0', strips spaces, converts invalid strings to NaN.
    """
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def safe_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Convert selected columns to numeric when present.
    """
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def detect_construction_cost_column(df: pd.DataFrame) -> str:
    """
    Auto-detect construction cost column from common candidates.
    """
    candidates = [
        "建造成本",
        "construction_cost",
        "construction_cost_2024",
        "build_cost",
        "build_cost_2024",
        "建设成本",
        "工程造价",
        "造价",
        "construction",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        "Construction-cost column could not be identified from parks.columns."
        " Set COL_CONSTRUCTION_COST to a valid source column."
    )

def assign_payback_bin(payback: pd.Series) -> pd.Series:
    """
    Assign payback bins; NaN -> N.A.
    """
    out = pd.cut(
        payback,
        bins=PAYBACK_BINS,
        labels=PAYBACK_LABELS,
        right=True,
        include_lowest=True
    )
    out = out.astype("object")
    out[pd.isna(payback)] = "N.A."
    return out

def median_finite(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors="coerce")
    s = s[np.isfinite(s)]
    return float(s.median()) if len(s) > 0 else np.nan

def mean_finite(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors="coerce")
    s = s[np.isfinite(s)]
    return float(s.mean()) if len(s) > 0 else np.nan

def format_scenario_name(m: float) -> str:
    if np.isclose(m, 1.0):
        return "Baseline"
    elif m < 1:
        return f"Low_{int(round((1 - m) * 100))}pct"
    else:
        return f"High_{int(round((m - 1) * 100))}pct"

def is_recoup_within_50y(x: str) -> bool:
    return x in {"0-5y", "5-20y", "20-50y"}

def get_type_label_column(visits_df: pd.DataFrame, parks_df: pd.DataFrame) -> str | None:
    """
    Decide which column to use as park type label for summary tables.
    Priority:
    1) manually specified cost label
    2) visit park_class_name
    3) cost park_type
    4) visit park_class
    """
    if COL_PARK_TYPE_LABEL_COST is not None and COL_PARK_TYPE_LABEL_COST in parks_df.columns:
        return COL_PARK_TYPE_LABEL_COST
    if COL_PARK_CLASS_NAME_VISIT in visits_df.columns:
        return COL_PARK_CLASS_NAME_VISIT
    if "park_type" in parks_df.columns:
        return "park_type"
    if COL_PARK_CLASS_VISIT in visits_df.columns:
        return COL_PARK_CLASS_VISIT
    return None

print("Reading visit files...")
visit_1 = pd.read_csv(VISIT_FILE_1)
visit_2 = pd.read_csv(VISIT_FILE_2)

print("Reading park cost file...")
parks = pd.read_excel(PARK_COST_FILE)

visits = pd.concat([visit_1, visit_2], ignore_index=True).copy()

print("\nVisit columns:")
print(visits.columns.tolist())

print("\nPark cost columns:")
print(parks.columns.tolist())

if COL_CONSTRUCTION_COST is None:
    COL_CONSTRUCTION_COST = detect_construction_cost_column(parks)
    print(f"\nAuto-detected construction cost column: {COL_CONSTRUCTION_COST}")
else:
    print(f"\nUsing manually specified construction cost column: {COL_CONSTRUCTION_COST}")

TYPE_LABEL_COL = get_type_label_column(visits, parks)
print(f"Type label source column: {TYPE_LABEL_COL}")

required_visit_cols = [COL_OSM_ID_VISIT, COL_AREA_VISIT, COL_STEPS_VISIT, COL_PARK_CLASS_VISIT]
for c in required_visit_cols:
    if c not in visits.columns:
        raise ValueError(f"The mobility file is missing required column: {c}")

required_cost_cols = [COL_OSM_ID_COST, COL_LAND_COST, COL_UNIT_MAINT, COL_CONSTRUCTION_COST]
for c in required_cost_cols:
    if c not in parks.columns:
        raise ValueError(f"The cost table is missing required column: {c}")

visits = visits.copy()
parks = parks.copy()

visits[COL_OSM_ID_VISIT] = standardize_osm_id(visits[COL_OSM_ID_VISIT])
parks[COL_OSM_ID_COST] = standardize_osm_id(parks[COL_OSM_ID_COST])

visit_numeric_cols = [COL_AREA_VISIT, COL_STEPS_VISIT]
cost_numeric_cols = [COL_LAND_COST, COL_UNIT_MAINT, COL_CONSTRUCTION_COST]

visits = safe_numeric(visits, visit_numeric_cols)
parks = safe_numeric(parks, cost_numeric_cols)

visits = visits.dropna(subset=[COL_OSM_ID_VISIT, COL_AREA_VISIT, COL_STEPS_VISIT]).copy()
visits = visits[visits[COL_STEPS_VISIT] >= 0].copy()

parks = parks.dropna(subset=[COL_OSM_ID_COST, COL_LAND_COST, COL_UNIT_MAINT, COL_CONSTRUCTION_COST]).copy()

print(f"\nVisits rows after cleaning: {len(visits):,}")
print(f"Parks rows after cleaning: {len(parks):,}")

agg_dict = {
    COL_AREA_VISIT: "first",
    COL_PARK_CLASS_VISIT: "first",
    COL_STEPS_VISIT: ["size", "sum"]
}

if COL_PARK_CLASS_NAME_VISIT in visits.columns:
    agg_dict[COL_PARK_CLASS_NAME_VISIT] = "first"

park_from_visits = visits.groupby(COL_OSM_ID_VISIT).agg(agg_dict)

park_from_visits.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in park_from_visits.columns
]
park_from_visits = park_from_visits.reset_index()

rename_map = {
    f"{COL_AREA_VISIT}_first": "area_m2_visit",
    f"{COL_PARK_CLASS_VISIT}_first": "park_class",
    f"{COL_STEPS_VISIT}_size": "n_visits",
    f"{COL_STEPS_VISIT}_sum": "steps_baseline_sum",
}
if f"{COL_PARK_CLASS_NAME_VISIT}_first" in park_from_visits.columns:
    rename_map[f"{COL_PARK_CLASS_NAME_VISIT}_first"] = "park_class_name"

park_from_visits = park_from_visits.rename(columns=rename_map)

park_df = park_from_visits.merge(
    parks,
    left_on=COL_OSM_ID_VISIT,
    right_on=COL_OSM_ID_COST,
    how="left"
).copy()

park_df = park_df.dropna(subset=[COL_LAND_COST, COL_CONSTRUCTION_COST, COL_UNIT_MAINT]).copy()

park_df["area_m2"] = pd.to_numeric(park_df["area_m2_visit"], errors="coerce")
park_df = park_df.dropna(subset=["area_m2"]).copy()

if TYPE_LABEL_COL == COL_PARK_CLASS_NAME_VISIT and "park_class_name" in park_df.columns:
    park_df["park_type_label"] = park_df["park_class_name"].astype(str)
elif TYPE_LABEL_COL is not None and TYPE_LABEL_COL in park_df.columns:
    park_df["park_type_label"] = park_df[TYPE_LABEL_COL].astype(str)
else:
    park_df["park_type_label"] = park_df["park_class"].astype(str)

park_df["annual_om_cost"] = park_df["area_m2"] * park_df[COL_UNIT_MAINT]

park_df["one_time_investment"] = park_df[COL_LAND_COST] + park_df[COL_CONSTRUCTION_COST]

park_df = park_df.dropna(
    subset=["steps_baseline_sum", "annual_om_cost", "one_time_investment"]
).copy()

print(f"\nParks retained for analysis after merge: {len(park_df):,}")
print(f"Unique parks in visits: {park_from_visits[COL_OSM_ID_VISIT].nunique():,}")
print(f"Unique parks after merging costs: {park_df[COL_OSM_ID_VISIT].nunique():,}")

visit_park_ids = set(park_from_visits[COL_OSM_ID_VISIT].dropna().astype(str))
merged_park_ids = set(park_df[COL_OSM_ID_VISIT].dropna().astype(str))
missing_after_merge = sorted(list(visit_park_ids - merged_park_ids))
print(f"Parks in visits but not retained after merge/cleaning: {len(missing_after_merge)}")

baseline_m = 1.0
baseline_name = format_scenario_name(baseline_m)

baseline_df = park_df.copy()
baseline_df["scenario_multiplier"] = baseline_m
baseline_df["scenario_name"] = baseline_name

baseline_df["steps_scaled_sum"] = baseline_df["steps_baseline_sum"] * baseline_m
baseline_df["annual_benefit_jpy"] = baseline_df["steps_scaled_sum"] * JPY_PER_STEP * EXPANSION_FACTOR
baseline_df["annual_net_benefit_jpy"] = baseline_df["annual_benefit_jpy"] - baseline_df["annual_om_cost"]

baseline_df["payback_years"] = np.where(
    baseline_df["annual_net_benefit_jpy"] > 0,
    baseline_df["one_time_investment"] / baseline_df["annual_net_benefit_jpy"],
    np.nan
)

baseline_df["payback_bin"] = assign_payback_bin(baseline_df["payback_years"])

baseline_bins = baseline_df[[COL_OSM_ID_VISIT, "payback_bin"]].rename(
    columns={"payback_bin": "baseline_payback_bin"}
).copy()

all_park_results = []
overall_rows = []
type_summary_rows = []
bin_shift_rows = []

# Scenarios vary inferred steps and associated benefits while holding investment and maintenance costs fixed.
for m in SCENARIOS:
    scenario_name = format_scenario_name(m)
    temp = park_df.copy()

    temp["scenario_multiplier"] = m
    temp["scenario_name"] = scenario_name

    temp["steps_scaled_sum"] = temp["steps_baseline_sum"] * m

    temp["annual_benefit_jpy"] = temp["steps_scaled_sum"] * JPY_PER_STEP * EXPANSION_FACTOR

    temp["annual_net_benefit_jpy"] = temp["annual_benefit_jpy"] - temp["annual_om_cost"]

    # This sensitivity module reports simple undiscounted payback from annual net benefit.
    temp["payback_years"] = np.where(
        temp["annual_net_benefit_jpy"] > 0,
        temp["one_time_investment"] / temp["annual_net_benefit_jpy"],
        np.nan
    )

    temp["payback_bin"] = assign_payback_bin(temp["payback_years"])

    keep_cols = [
        COL_OSM_ID_VISIT,
        "park_class",
        "park_type_label",
        "area_m2",
        "n_visits",
        "steps_baseline_sum",
        "steps_scaled_sum",
        COL_LAND_COST,
        COL_CONSTRUCTION_COST,
        "annual_om_cost",
        "one_time_investment",
        "annual_benefit_jpy",
        "annual_net_benefit_jpy",
        "payback_years",
        "payback_bin",
        "scenario_multiplier",
        "scenario_name",
    ]
    all_park_results.append(temp[keep_cols].copy())

    n_total = len(temp)
    n_0_5 = (temp["payback_bin"] == "0-5y").sum()
    n_5_20 = (temp["payback_bin"] == "5-20y").sum()
    n_20_50 = (temp["payback_bin"] == "20-50y").sum()
    n_gt_50 = (temp["payback_bin"] == ">50y").sum()
    n_na = (temp["payback_bin"] == "N.A.").sum()
    n_recoup_50 = n_0_5 + n_5_20 + n_20_50

    overall_rows.append({
        "scenario_name": scenario_name,
        "scenario_multiplier": m,
        "n_parks": n_total,
        "total_annual_benefit_jpy": temp["annual_benefit_jpy"].sum(),
        "total_annual_net_benefit_jpy": temp["annual_net_benefit_jpy"].sum(),
        "share_0_5y": n_0_5 / n_total if n_total else np.nan,
        "share_5_20y": n_5_20 / n_total if n_total else np.nan,
        "share_20_50y": n_20_50 / n_total if n_total else np.nan,
        "share_gt_50y": n_gt_50 / n_total if n_total else np.nan,
        "share_na": n_na / n_total if n_total else np.nan,
        "share_recoup_within_50y": n_recoup_50 / n_total if n_total else np.nan,
        "median_finite_payback_years": median_finite(temp["payback_years"]),
        "mean_finite_payback_years": mean_finite(temp["payback_years"]),
    })

    for ptype, g in temp.groupby("park_type_label", dropna=False):
        n = len(g)
        c0 = (g["payback_bin"] == "0-5y").sum()
        c1 = (g["payback_bin"] == "5-20y").sum()
        c2 = (g["payback_bin"] == "20-50y").sum()
        c3 = (g["payback_bin"] == ">50y").sum()
        c4 = (g["payback_bin"] == "N.A.").sum()

        type_summary_rows.append({
            "scenario_name": scenario_name,
            "scenario_multiplier": m,
            "park_type_label": ptype,
            "n_parks": n,
            "share_0_5y": c0 / n if n else np.nan,
            "share_5_20y": c1 / n if n else np.nan,
            "share_20_50y": c2 / n if n else np.nan,
            "share_gt_50y": c3 / n if n else np.nan,
            "share_na": c4 / n if n else np.nan,
            "share_recoup_within_50y": (c0 + c1 + c2) / n if n else np.nan,
            "median_finite_payback_years": median_finite(g["payback_years"]),
            "mean_finite_payback_years": mean_finite(g["payback_years"]),
            "total_annual_benefit_jpy": g["annual_benefit_jpy"].sum(),
            "median_annual_benefit_jpy": g["annual_benefit_jpy"].median(),
        })

    if not np.isclose(m, 1.0):
        compare = temp[[COL_OSM_ID_VISIT, "payback_bin"]].merge(
            baseline_bins,
            on=COL_OSM_ID_VISIT,
            how="left"
        ).copy()

        compare["changed_bin"] = compare["payback_bin"] != compare["baseline_payback_bin"]
        compare["current_le_50"] = compare["payback_bin"].apply(is_recoup_within_50y)
        compare["base_le_50"] = compare["baseline_payback_bin"].apply(is_recoup_within_50y)

        changed_n = compare["changed_bin"].sum()
        total_n = len(compare)
        worse_n = ((compare["base_le_50"] == True) & (compare["current_le_50"] == False)).sum()
        better_n = ((compare["base_le_50"] == False) & (compare["current_le_50"] == True)).sum()

        bin_shift_rows.append({
            "scenario_name": scenario_name,
            "scenario_multiplier": m,
            "n_parks": total_n,
            "n_changed_payback_bin": changed_n,
            "share_changed_payback_bin": changed_n / total_n if total_n else np.nan,
            "n_le50_to_gt50_or_na": worse_n,
            "n_gt50_or_na_to_le50": better_n,
        })

bin_shift_rows.insert(0, {
    "scenario_name": "Baseline",
    "scenario_multiplier": 1.0,
    "n_parks": len(baseline_df),
    "n_changed_payback_bin": 0,
    "share_changed_payback_bin": 0.0,
    "n_le50_to_gt50_or_na": 0,
    "n_gt50_or_na_to_le50": 0,
})

park_results_all = pd.concat(all_park_results, ignore_index=True)
overall_summary = pd.DataFrame(overall_rows).sort_values("scenario_multiplier").reset_index(drop=True)
type_summary = pd.DataFrame(type_summary_rows).sort_values(
    ["scenario_multiplier", "park_type_label"]
).reset_index(drop=True)
bin_shift_summary = pd.DataFrame(bin_shift_rows).sort_values("scenario_multiplier").reset_index(drop=True)

baseline_rank = (
    park_results_all[park_results_all["scenario_multiplier"] == 1.0]
    [[COL_OSM_ID_VISIT, "annual_benefit_jpy"]]
    .sort_values("annual_benefit_jpy", ascending=False)
    .reset_index(drop=True)
    .copy()
)
baseline_rank["rank_baseline"] = np.arange(1, len(baseline_rank) + 1)

ranking_rows = []
for m in SCENARIOS:
    temp_rank = (
        park_results_all[park_results_all["scenario_multiplier"] == m]
        [[COL_OSM_ID_VISIT, "annual_benefit_jpy"]]
        .sort_values("annual_benefit_jpy", ascending=False)
        .reset_index(drop=True)
        .copy()
    )
    temp_rank["rank_current"] = np.arange(1, len(temp_rank) + 1)

    chk = temp_rank.merge(
        baseline_rank[[COL_OSM_ID_VISIT, "rank_baseline"]],
        on=COL_OSM_ID_VISIT,
        how="left"
    ).copy()

    ranking_rows.append({
        "scenario_name": format_scenario_name(m),
        "scenario_multiplier": m,
        "park_benefit_ranking_identical_to_baseline": bool(
            (chk["rank_current"] == chk["rank_baseline"]).all()
        )
    })

ranking_check = pd.DataFrame(ranking_rows).sort_values("scenario_multiplier").reset_index(drop=True)

type_summary_pivot = type_summary.pivot_table(
    index="park_type_label",
    columns="scenario_name",
    values=[
        "share_0_5y",
        "share_5_20y",
        "share_20_50y",
        "share_gt_50y",
        "share_na",
        "share_recoup_within_50y",
        "median_finite_payback_years",
    ],
    aggfunc="first"
)

excel_out = OUTPUT_DIR / "sensitivity_analysis_steps_payback.xlsx"
csv_overall = OUTPUT_DIR / "overall_summary.csv"
csv_type = OUTPUT_DIR / "type_summary.csv"
csv_shift = OUTPUT_DIR / "bin_shift_summary.csv"
csv_park = OUTPUT_DIR / "park_results_all_scenarios.csv"
csv_rank = OUTPUT_DIR / "ranking_check.csv"
txt_info = OUTPUT_DIR / "readme_outputs.txt"

overall_summary.to_csv(csv_overall, index=False, encoding="utf-8-sig")
type_summary.to_csv(csv_type, index=False, encoding="utf-8-sig")
bin_shift_summary.to_csv(csv_shift, index=False, encoding="utf-8-sig")
park_results_all.to_csv(csv_park, index=False, encoding="utf-8-sig")
ranking_check.to_csv(csv_rank, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
    overall_summary.to_excel(writer, sheet_name="overall_summary", index=False)
    type_summary.to_excel(writer, sheet_name="type_summary_long", index=False)
    type_summary_pivot.to_excel(writer, sheet_name="type_summary_pivot")
    bin_shift_summary.to_excel(writer, sheet_name="bin_shift_summary", index=False)
    ranking_check.to_excel(writer, sheet_name="ranking_check", index=False)
    park_results_all.to_excel(writer, sheet_name="park_results_all", index=False)

with open(txt_info, "w", encoding="utf-8") as f:
    f.write("This folder contains only derived outputs.\n")
    f.write("Original input files were read but not modified.\n")
    f.write(f"VISIT_FILE_1 = {VISIT_FILE_1}\n")
    f.write(f"VISIT_FILE_2 = {VISIT_FILE_2}\n")
    f.write(f"PARK_COST_FILE = {PARK_COST_FILE}\n")
    f.write(f"Construction cost column used = {COL_CONSTRUCTION_COST}\n")
    f.write(f"Type label source = {TYPE_LABEL_COL}\n")
    f.write(f"Scenarios = {SCENARIOS}\n")
    f.write(f"Expansion factor = {EXPANSION_FACTOR}\n")
    f.write(f"JPY per step = {JPY_PER_STEP}\n")

print("\nDone.")
print(f"All outputs written to: {OUTPUT_DIR}")
print(f"Excel file: {excel_out}")

print("\n===== overall_summary =====")
display(overall_summary)

print("\n===== bin_shift_summary =====")
display(bin_shift_summary)

print("\n===== ranking_check =====")
display(ranking_check)

print("\n===== type_summary (first 20 rows) =====")
display(type_summary.head(20))

print("\nQuick notes:")
print("1) Original source files were NOT modified.")
print("2) Check overall_summary first: share_recoup_within_50y, share_na, median_finite_payback_years.")
print("3) Check bin_shift_summary: how many parks crossed payback bins relative to baseline.")
print("4) ranking_check should normally be all True.")
